# Module 2: Information Gain, Leakage-Free Scaling, PCA & Fisher LDA

**Z3003 Data Mining: Scaling to Large Data (BS Core) · IIT Madras Zanzibar**

Entropy split selection, standardized vs. raw PCA, and linear discriminant projection.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/inyalala/iitmz-teaching-site/blob/main/courses/dm-scale/labs/DM02_Feature_Selection_PCA_LDA.ipynb)

Run cells top to bottom (`Runtime > Run all`). Every method is self-contained and verified with assertions.

## 1. Entropy, Information Gain & Leakage-Free Normalization

Supervised discretization picks the midpoint maximizing Information Gain; scaling parameters $(\mu, \sigma)$ are fitted strictly on training records.

In [1]:
import math
import numpy as np
from collections import Counter


def entropy(labels):
    n = len(labels)
    return -sum((c / n) * math.log2(c / n) for c in Counter(labels).values())


def best_split_ig(vals, labels):
    order = np.argsort(vals)
    sv, sl = vals[order], labels[order]
    base, n = entropy(sl.tolist()), len(sl)
    best_t, best_ig = None, -1.0
    for i in range(n - 1):
        if sv[i] == sv[i + 1]:
            continue
        t = 0.5 * float(sv[i] + sv[i + 1])
        w_h = ((i + 1) / n) * entropy(sl[: i + 1].tolist()) + ((n - i - 1) / n) * entropy(sl[i + 1 :].tolist())
        if base - w_h > best_ig:
            best_t, best_ig = t, base - w_h
    return best_t, best_ig


rain = np.array([1000, 1100, 1200, 1300, 1400, 1500, 1600, 1700], dtype=float)
lbls = np.array(["L", "L", "L", "H", "H", "H", "H", "L"])
t, ig = best_split_ig(rain, lbls)
print(f"Best rainfall threshold = {t:.0f} mm (IG = {ig:.3f} bits)")
assert abs(t - 1250.0) < 1e-6


Best rainfall threshold = 1250 mm (IG = 0.549 bits)


## 2. PCA Eigendecomposition vs. Unscaled Artifact & Fisher LDA

On raw Zanzibar farm columns (`revenue_usd` vs `farm_size_ha`), unscaled PCA gives a fake 99% PC1 dominated by currency units. Standardizing first yields honest component variance shares.

In [2]:
rng = np.random.default_rng(3003)
n = 300
farm_size = rng.uniform(0.3, 6.0, n)
rainfall = rng.normal(1400, 220, n)
yield_t = 0.4 * farm_size + 0.002 * rainfall + rng.normal(0, 0.4, n)
revenue_usd = yield_t * 1000 * 3.5

X_raw = np.column_stack([farm_size, rainfall, yield_t, revenue_usd])
X_std = (X_raw - X_raw.mean(axis=0)) / X_raw.std(axis=0)

for name, mat in [("Raw units", X_raw), ("Standardized", X_std)]:
    ev, _ = np.linalg.eigh(np.cov(mat, rowvar=False))
    ev = np.sort(ev)[::-1]
    ratios = np.round(100 * ev / ev.sum(), 1)
    print(f"{name:12s} variance %: {ratios.tolist()}")

# Fisher LDA projection on two farm yield classes
y_cls = (yield_t > np.median(yield_t)).astype(int)
X0, X1 = X_std[y_cls == 0], X_std[y_cls == 1]
m0, m1 = X0.mean(axis=0), X1.mean(axis=0)
Sw = (X0 - m0).T @ (X0 - m0) + (X1 - m1).T @ (X1 - m1) + 1e-6 * np.eye(4)
w_lda = np.linalg.solve(Sw, m1 - m0)
w_lda /= np.linalg.norm(w_lda)
print("Fisher LDA unit direction:", np.round(w_lda, 3).tolist())


Raw units    variance %: [99.6, 0.4, 0.0, 0.0]
Standardized variance %: [68.3, 28.2, 3.5, -0.0]
Fisher LDA unit direction: [0.271, -0.044, 0.68, 0.68]
